In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings

# Suppress DeprecationWarning for jupyter_client
warnings.filterwarnings("ignore", category=DeprecationWarning, module='jupyter_client')

In [7]:
# Load the dataset
df = pd.read_csv('anime.csv')

In [8]:
# Handle missing values
df['genre'] = df['genre'].fillna('Unknown')
df['type'] = df['type'].fillna('Unknown')
df['rating'] = df['rating'].fillna(df['rating'].median())

In [9]:
# Clean 'episodes' column (replace 'Unknown' with median)
df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce') # Added errors='coerce'
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

In [10]:
# Use TF-IDF to convert genres into numerical vectors
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['genre'])

In [11]:
# Normalize numerical features (rating and members)
scaler = MinMaxScaler()
numerical_features = scaler.fit_transform(df[['rating', 'members']])

In [12]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [13]:
def get_recommendations(title, cosine_sim=cosine_sim, n=10):
    # Get index of the anime that matches the title
    if title not in df['name'].values:
        return "Anime not found."

    idx = df[df['name'] == title].index[0]

    # Get pairwise similarity scores of all anime with that anime
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the anime based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the n most similar anime (skip the first one as it is the anime itself)
    sim_scores = sim_scores[1:n+1]

    # Get the anime indices
    anime_indices = [i[0] for i in sim_scores]

    # Return the top n most similar anime
    return df['name'].iloc[anime_indices]

In [14]:
# Test the system
print("Recommendations for 'Steins;Gate':")
print(get_recommendations('Steins;Gate'))

Recommendations for 'Steins;Gate':
59              Steins;Gate Movie: Fuka Ryouiki no Déjà vu
126                  Steins;Gate: Oukoubakko no Poriomania
196      Steins;Gate: Kyoukaimenjou no Missing Link - D...
10898                                        Steins;Gate 0
5126                                         Under the Dog
5525                                          Loups=Garous
6889                                    Loups=Garous Pilot
5249                           Kyoto Animation: Megane-hen
2518                                           Ibara no Ou
5454                                              Duan Nao
Name: name, dtype: object


In [15]:
def evaluate_recommender(test_indices, n_recs=10):
    precisions = []
    recalls = []

    for idx in test_indices:
        target_genres = set(df.iloc[idx]['genre'].split(', '))
        # Get recommendations
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n_recs+1]
        rec_indices = [i[0] for i in sim_scores]

        # Calculate relevance
        relevant_count = 0
        for r_idx in rec_indices:
            rec_genres = set(df.iloc[r_idx]['genre'].split(', '))
            # If intersection is significant, count as relevant
            if len(target_genres.intersection(rec_genres)) / len(target_genres) >= 0.5:
                relevant_count += 1

        precisions.append(relevant_count / n_recs)
        # Simplified recall: assuming there are at least n_recs relevant items in the whole set
        recalls.append(relevant_count / n_recs)

    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    f1 = 2 * (avg_p * avg_r) / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0

    return avg_p, avg_r, f1

In [16]:
# Split dataset indices for evaluation
train_idx, test_idx = train_test_split(range(len(df)), test_size=0.2, random_state=42)

# Evaluate on a sample of the test set for efficiency
sample_test_idx = np.random.choice(test_idx, 100)
p, r, f = evaluate_recommender(sample_test_idx)

print(f"\nEvaluation Results (Sample of 100 Test Items):")
print(f"Precision @ 10: {p:.4f}")
print(f"Recall @ 10: {r:.4f}")
print(f"F1-Score: {f:.4f}")


Evaluation Results (Sample of 100 Test Items):
Precision @ 10: 0.9930
Recall @ 10: 0.9930
F1-Score: 0.9930
